In [1]:
from pathlib import Path
import pandas as pd

print("02.24 STEP 1 - INSPECT HISTORICAL MONTHLY CSVs")
print("=" * 65)

BASE_DIR = Path.cwd().parent

SALES_DIR = BASE_DIR / "data" / "synthetic"

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

print(f"\nSales files found: {len(sales_files)}")

print("\nFILE LIST")
print("-" * 65)

for file in sales_files:
    print(file.name)

print("\nMONTHLY ROW COUNTS")
print("-" * 65)

total_rows = 0

for file in sales_files:
    df = pd.read_csv(file)
    row_count = len(df)
    total_rows += row_count

    print(f"{file.name}: {row_count:,} rows")

print("\nHISTORICAL RANGE")
print("-" * 65)

if sales_files:
    print(f"First file: {sales_files[0].name}")
    print(f"Last file:  {sales_files[-1].name}")

print(f"\nTotal historical rows: {total_rows:,}")

print("\n02.24 Step 1 inspection: COMPLETED")

02.24 STEP 1 - INSPECT HISTORICAL MONTHLY CSVs

Sales files found: 32

FILE LIST
-----------------------------------------------------------------
sales_2024-01.csv
sales_2024-02.csv
sales_2024-03.csv
sales_2024-04.csv
sales_2024-05.csv
sales_2024-06.csv
sales_2024-07.csv
sales_2024-08.csv
sales_2024-09.csv
sales_2024-10.csv
sales_2024-11.csv
sales_2024-12.csv
sales_2025-01.csv
sales_2025-02.csv
sales_2025-03.csv
sales_2025-04.csv
sales_2025-05.csv
sales_2025-06.csv
sales_2025-07.csv
sales_2025-08.csv
sales_2025-09.csv
sales_2025-10.csv
sales_2025-11.csv
sales_2025-12.csv
sales_2026-01.csv
sales_2026-02.csv
sales_2026-03.csv
sales_2026-04.csv
sales_2026-05.csv
sales_2026-06.csv
sales_2026-07.csv
sales_2026-08.csv

MONTHLY ROW COUNTS
-----------------------------------------------------------------
sales_2024-01.csv: 53,200 rows
sales_2024-02.csv: 50,400 rows
sales_2024-03.csv: 56,000 rows
sales_2024-04.csv: 57,100 rows
sales_2024-05.csv: 58,800 rows
sales_2024-06.csv: 60,500 rows
sales

In [2]:
from pathlib import Path
import pandas as pd

print("02.24 STEP 2 - VALIDATE MONTHLY COVERAGE")
print("=" * 65)

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

# Required historical period
expected_months = pd.period_range(
    start="2024-01",
    end="2026-08",
    freq="M"
)

expected_files = [
    f"sales_{period.strftime('%Y-%m')}.csv"
    for period in expected_months
]

actual_files = sorted(
    file.name
    for file in SALES_DIR.glob("sales_*.csv")
)

print("\nEXPECTED MONTHS")
print("-" * 65)

print(f"Expected monthly files: {len(expected_files)}")
print(f"Actual monthly files:   {len(actual_files)}")

missing_files = sorted(
    set(expected_files) - set(actual_files)
)

unexpected_files = sorted(
    set(actual_files) - set(expected_files)
)

print("\nMISSING FILES")
print("-" * 65)

if missing_files:
    for file in missing_files:
        print(file)
else:
    print("None")

print("\nUNEXPECTED FILES")
print("-" * 65)

if unexpected_files:
    for file in unexpected_files:
        print(file)
else:
    print("None")

# Validate exact monthly sequence
monthly_sequence_valid = (
    len(actual_files) == len(expected_files)
    and actual_files == expected_files
)

print("\nMONTHLY COVERAGE VALIDATION")
print("-" * 65)

print(f"First required month: {expected_files[0]}")
print(f"Last required month:  {expected_files[-1]}")
print(f"Monthly sequence valid: {monthly_sequence_valid}")

if monthly_sequence_valid:
    print("\n02.24 Step 2 validation: PASSED")
else:
    print("\n02.24 Step 2 validation: NEEDS REVIEW")

02.24 STEP 2 - VALIDATE MONTHLY COVERAGE

EXPECTED MONTHS
-----------------------------------------------------------------
Expected monthly files: 32
Actual monthly files:   32

MISSING FILES
-----------------------------------------------------------------
None

UNEXPECTED FILES
-----------------------------------------------------------------
None

MONTHLY COVERAGE VALIDATION
-----------------------------------------------------------------
First required month: sales_2024-01.csv
Last required month:  sales_2026-08.csv
Monthly sequence valid: True

02.24 Step 2 validation: PASSED


In [3]:
from pathlib import Path
import pandas as pd

print("02.24 STEP 3 - VALIDATE FILE MONTH AGAINST ORDER DATE")
print("=" * 65)

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

total_rows = 0
files_valid = 0
files_invalid = 0

invalid_files = []

for file in sales_files:

    df = pd.read_csv(file)

    total_rows += len(df)

    # Expected month from filename
    expected_period = pd.Period(
        file.stem.replace("sales_", ""),
        freq="M"
    )

    # Convert order_date
    order_dates = pd.to_datetime(
        df["order_date"],
        errors="coerce"
    )

    actual_periods = order_dates.dt.to_period("M")

    # Check whether every valid date belongs to expected month
    invalid_count = (
        actual_periods.notna()
        & (actual_periods != expected_period)
    ).sum()

    invalid_date_count = order_dates.isna().sum()

    if invalid_count == 0 and invalid_date_count == 0:
        files_valid += 1
    else:
        files_invalid += 1

        invalid_files.append({
            "file": file.name,
            "expected_month": str(expected_period),
            "wrong_month_rows": int(invalid_count),
            "invalid_dates": int(invalid_date_count)
        })

print("\nVALIDATION SUMMARY")
print("-" * 65)

print(f"Files checked: {len(sales_files)}")
print(f"Files valid: {files_valid}")
print(f"Files invalid: {files_invalid}")
print(f"Total rows checked: {total_rows:,}")

print("\nINVALID FILES")
print("-" * 65)

if invalid_files:
    for item in invalid_files:
        print(item)
else:
    print("None")

print("\nFINAL VALIDATION")
print("-" * 65)

month_alignment_valid = (
    len(sales_files) == 32
    and total_rows == 2_118_200
    and files_invalid == 0
)

print(f"Month alignment valid: {month_alignment_valid}")

if month_alignment_valid:
    print("\n02.24 Step 3 validation: PASSED")
else:
    print("\n02.24 Step 3 validation: NEEDS REVIEW")

02.24 STEP 3 - VALIDATE FILE MONTH AGAINST ORDER DATE

VALIDATION SUMMARY
-----------------------------------------------------------------
Files checked: 32
Files valid: 32
Files invalid: 0
Total rows checked: 2,118,200

INVALID FILES
-----------------------------------------------------------------
None

FINAL VALIDATION
-----------------------------------------------------------------
Month alignment valid: True

02.24 Step 3 validation: PASSED


In [4]:
from pathlib import Path
import pandas as pd

print("02.24 STEP 4 - VALIDATE HISTORICAL ROW COUNT")
print("=" * 65)

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

EXPECTED_FILES = 32
EXPECTED_ROWS = 2_118_200

total_rows = 0
file_row_counts = []

for file in sales_files:
    df = pd.read_csv(file)

    row_count = len(df)
    total_rows += row_count

    file_row_counts.append({
        "file": file.name,
        "rows": row_count
    })

print("\nROW COUNT SUMMARY")
print("-" * 65)

print(f"Expected files: {EXPECTED_FILES}")
print(f"Actual files: {len(sales_files)}")
print(f"Expected total rows: {EXPECTED_ROWS:,}")
print(f"Actual total rows: {total_rows:,}")

print("\nMONTHLY ROW COUNTS")
print("-" * 65)

for item in file_row_counts:
    print(f"{item['file']}: {item['rows']:,}")

file_count_valid = len(sales_files) == EXPECTED_FILES
row_count_valid = total_rows == EXPECTED_ROWS

print("\nFINAL VALIDATION")
print("-" * 65)

print(f"File count valid: {file_count_valid}")
print(f"Total row count valid: {row_count_valid}")

if file_count_valid and row_count_valid:
    print("\n02.24 Step 4 validation: PASSED")
else:
    print("\n02.24 Step 4 validation: NEEDS REVIEW")

02.24 STEP 4 - VALIDATE HISTORICAL ROW COUNT

ROW COUNT SUMMARY
-----------------------------------------------------------------
Expected files: 32
Actual files: 32
Expected total rows: 2,118,200
Actual total rows: 2,118,200

MONTHLY ROW COUNTS
-----------------------------------------------------------------
sales_2024-01.csv: 53,200
sales_2024-02.csv: 50,400
sales_2024-03.csv: 56,000
sales_2024-04.csv: 57,100
sales_2024-05.csv: 58,800
sales_2024-06.csv: 60,500
sales_2024-07.csv: 57,700
sales_2024-08.csv: 60,500
sales_2024-09.csv: 64,400
sales_2024-10.csv: 75,600
sales_2024-11.csv: 81,200
sales_2024-12.csv: 89,600
sales_2025-01.csv: 57,500
sales_2025-02.csv: 54,400
sales_2025-03.csv: 60,500
sales_2025-04.csv: 61,700
sales_2025-05.csv: 63,500
sales_2025-06.csv: 65,300
sales_2025-07.csv: 62,300
sales_2025-08.csv: 65,300
sales_2025-09.csv: 69,600
sales_2025-10.csv: 81,600
sales_2025-11.csv: 87,700
sales_2025-12.csv: 96,800
sales_2026-01.csv: 61,700
sales_2026-02.csv: 58,500
sales_2026-0

In [5]:
from pathlib import Path
import pandas as pd

print("02.24 STEP 5 - FINAL HISTORICAL CSV VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

EXPECTED_FILES = 32
EXPECTED_ROWS = 2_118_200

REQUIRED_COLUMNS = [
    "sales_line_id",
    "order_id",
    "order_date",
    "order_time",
    "customer_id",
    "product_id",
    "store_id",
    "salesperson_id",
    "promotion_id",
    "sales_channel",
    "quantity",
    "unit_selling_price",
    "discount_pct",
    "payment_method",
    "order_status"
]

total_rows = 0
schema_consistent = True
required_columns_present = True
duplicate_rows = 0

reference_columns = None

for file in sales_files:

    df = pd.read_csv(file)

    total_rows += len(df)

    # Schema consistency
    current_columns = list(df.columns)

    if reference_columns is None:
        reference_columns = current_columns
    elif current_columns != reference_columns:
        schema_consistent = False

    # Required columns
    missing_columns = set(REQUIRED_COLUMNS) - set(current_columns)

    if missing_columns:
        required_columns_present = False
        print(f"Missing columns in {file.name}: {sorted(missing_columns)}")

    # Duplicate complete rows
    duplicate_rows += int(df.duplicated().sum())


file_count_valid = len(sales_files) == EXPECTED_FILES
row_count_valid = total_rows == EXPECTED_ROWS
no_duplicates = duplicate_rows == 0

print("\nFINAL VALIDATION")
print("-" * 65)

print(f"Files found: {len(sales_files)}")
print(f"Expected files: {EXPECTED_FILES}")

print(f"\nTotal rows: {total_rows:,}")
print(f"Expected rows: {EXPECTED_ROWS:,}")

print(f"\nSchema consistent: {schema_consistent}")
print(f"Required columns present: {required_columns_present}")
print(f"Duplicate complete rows: {duplicate_rows}")

print("\nVALIDATION STATUS")
print("-" * 65)

print(f"File count valid: {file_count_valid}")
print(f"Row count valid: {row_count_valid}")
print(f"Schema valid: {schema_consistent}")
print(f"Required columns valid: {required_columns_present}")
print(f"No duplicate complete rows: {no_duplicates}")

if (
    file_count_valid
    and row_count_valid
    and schema_consistent
    and required_columns_present
    and no_duplicates
):
    print("\n02.24 Step 5 validation: PASSED")
else:
    print("\n02.24 Step 5 validation: NEEDS REVIEW")

02.24 STEP 5 - FINAL HISTORICAL CSV VALIDATION

FINAL VALIDATION
-----------------------------------------------------------------
Files found: 32
Expected files: 32

Total rows: 2,118,200
Expected rows: 2,118,200

Schema consistent: True
Required columns present: True
Duplicate complete rows: 0

VALIDATION STATUS
-----------------------------------------------------------------
File count valid: True
Row count valid: True
Schema valid: True
Required columns valid: True
No duplicate complete rows: True

02.24 Step 5 validation: PASSED


In [6]:
from pathlib import Path
import pandas as pd

print("02.24 STEP 6 - SAVE HISTORICAL CSV VALIDATION SUMMARY")
print("=" * 65)

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"
OUTPUT_DIR = SALES_DIR / "historical_monthly_validation"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

EXPECTED_FILES = 32
EXPECTED_ROWS = 2_118_200

total_rows = 0
validation_rows = []

for file in sales_files:
    df = pd.read_csv(file)

    row_count = len(df)
    total_rows += row_count

    validation_rows.append({
        "file_name": file.name,
        "row_count": row_count
    })

validation_df = pd.DataFrame(validation_rows)

summary_df = pd.DataFrame([{
    "expected_files": EXPECTED_FILES,
    "actual_files": len(sales_files),
    "expected_rows": EXPECTED_ROWS,
    "actual_rows": total_rows,
    "file_count_valid": len(sales_files) == EXPECTED_FILES,
    "row_count_valid": total_rows == EXPECTED_ROWS
}])

detail_file = OUTPUT_DIR / "historical_monthly_row_validation.csv"
summary_file = OUTPUT_DIR / "historical_monthly_validation_summary.csv"

validation_df.to_csv(detail_file, index=False)
summary_df.to_csv(summary_file, index=False)

print(f"Detail validation saved to:")
print(detail_file)

print(f"\nSummary validation saved to:")
print(summary_file)

print("\nFINAL RESULT")
print("-" * 65)

if (
    len(sales_files) == EXPECTED_FILES
    and total_rows == EXPECTED_ROWS
):
    print("02.24 Step 6: PASSED")
    print("Historical monthly CSV validation summary saved successfully.")
else:
    print("02.24 Step 6: NEEDS REVIEW")

02.24 STEP 6 - SAVE HISTORICAL CSV VALIDATION SUMMARY
Detail validation saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\historical_monthly_validation\historical_monthly_row_validation.csv

Summary validation saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\historical_monthly_validation\historical_monthly_validation_summary.csv

FINAL RESULT
-----------------------------------------------------------------
02.24 Step 6: PASSED
Historical monthly CSV validation summary saved successfully.
